# 00 · Setup: create folders and extract the zip files

**What this notebook does**
1. Checks that the Python packages are installed.
2. Creates the project folders (`data/raw`, `data/interim`, `data/processed`, `reports/figures`).
3. Finds your two zip files in `data/zips/`, works out which one is which, and extracts them into the right folders.
4. Checks that every expected CSV file is there.

**Before running:** put both zip files (the incident one and `sales.zip`) inside the folder `opsinsight/data/zips/`.
The file names don't matter, this notebook looks *inside* each zip to decide where it goes.

**How to run:** click **Run All** at the top (or press `Shift + Enter` on each cell, one by one).

## Step 1: Check the packages are installed
If this cell shows an error like `ModuleNotFoundError`, go back to the terminal and run `pip install -r requirements.txt`.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import seaborn
import sklearn
import lightgbm
import pyarrow

print("Python    :", sys.version.split()[0])
print("pandas    :", pd.__version__)
print("numpy     :", np.__version__)
print("sklearn   :", sklearn.__version__)
print("lightgbm  :", lightgbm.__version__)
print("All packages found ✔")

Python    : 3.13.5
pandas    : 2.3.3
numpy     : 2.5.4
sklearn   : 1.9.1
lightgbm  : 4.7.0
All packages found ✔


## Step 2: Find the project root folder
Notebooks live in `opsinsight/notebooks/`, but the data lives in `opsinsight/data/`.
This small loop walks up from the current folder until it finds the folder that contains `requirements.txt`, that is the project root.
Every notebook starts with this same cell so paths always work.

In [2]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print("Project root:", ROOT)

Project root: c:\Users\Arpita Thakur\Desktop\opsinsight 2\opsinsight 2\OpsInsight


## Step 3: Create the folders

| Folder | What goes in it |
|---|---|
| `data/zips/` | the original zip files you downloaded |
| `data/raw/incidents/`, `data/raw/sales/` | the extracted CSVs, **never edit these** |
| `data/interim/` | cleaned tables (after notebooks 02 and 05) |
| `data/processed/` | final model-ready feature tables (after notebooks 03 and 06) |
| `reports/figures/` | charts saved by the EDA notebooks |
| `reports/` | data quality reports and feature dictionaries |

`exist_ok=True` means "don't complain if the folder already exists", so it's safe to run this again.

In [3]:
folders = [
    "data/zips",
    "data/raw/incidents",
    "data/raw/sales",
    "data/interim",
    "data/processed",
    "reports/figures",
]
for f in folders:
    (ROOT / f).mkdir(parents=True, exist_ok=True)
    print("ready:", f)

ready: data/zips
ready: data/raw/incidents
ready: data/raw/sales
ready: data/interim
ready: data/processed
ready: reports/figures


## Step 4: Extract the zip files into the right place
We open each zip and look at the names of the files inside:
- if it contains `incident_event_log.csv` → it goes to `data/raw/incidents/`
- if it contains files starting with `olist_` → it goes to `data/raw/sales/`

In [4]:
import zipfile

zips = sorted((ROOT / "data/zips").glob("*.zip"))
print("Zip files found:", [z.name for z in zips])
if not zips:
    raise FileNotFoundError("No zip files found. Copy both zip files into opsinsight/data/zips/ and run again.")

for z in zips:
    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        if any(n.endswith("incident_event_log.csv") for n in names):
            target = ROOT / "data/raw/incidents"
        elif any(Path(n).name.startswith("olist_") for n in names):
            target = ROOT / "data/raw/sales"
        else:
            print(f"Skipping {z.name}: don't recognise its contents")
            continue
        # extract only the CSV files, flattening any sub-folders inside the zip
        for n in names:
            if n.lower().endswith(".csv") and not n.startswith("__MACOSX"):
                out = target / Path(n).name
                with zf.open(n) as src, open(out, "wb") as dst:
                    dst.write(src.read())
        print(f"{z.name}  →  {target.relative_to(ROOT)}")

Zip files found: ['95a58d4c-incidentmanagementprocessenrichedeventlog.zip', 'sales.zip']
95a58d4c-incidentmanagementprocessenrichedeventlog.zip  →  data\raw\incidents
sales.zip  →  data\raw\sales


## Step 5: Check every expected file is there

In [5]:
expected = {
    "data/raw/incidents": ["incident_event_log.csv"],
    "data/raw/sales": [
        "olist_orders_dataset.csv",
        "olist_order_items_dataset.csv",
        "olist_order_payments_dataset.csv",
        "olist_order_reviews_dataset.csv",
        "olist_customers_dataset.csv",
        "olist_sellers_dataset.csv",
        "olist_products_dataset.csv",
        "olist_geolocation_dataset.csv",
        "product_category_name_translation.csv",
    ],
}
missing = []
for folder, files in expected.items():
    for f in files:
        p = ROOT / folder / f
        if p.exists():
            print(f"✔ {folder}/{f}  ({p.stat().st_size / 1e6:.1f} MB)")
        else:
            print(f"✘ MISSING: {folder}/{f}")
            missing.append(f)

print("\nAll files ready, go to notebook 01." if not missing else f"\n{len(missing)} file(s) missing, check your zips.")

✔ data/raw/incidents/incident_event_log.csv  (46.2 MB)
✔ data/raw/sales/olist_orders_dataset.csv  (17.7 MB)
✔ data/raw/sales/olist_order_items_dataset.csv  (15.4 MB)
✔ data/raw/sales/olist_order_payments_dataset.csv  (5.8 MB)
✔ data/raw/sales/olist_order_reviews_dataset.csv  (14.5 MB)
✔ data/raw/sales/olist_customers_dataset.csv  (9.0 MB)
✔ data/raw/sales/olist_sellers_dataset.csv  (0.2 MB)
✔ data/raw/sales/olist_products_dataset.csv  (2.4 MB)
✔ data/raw/sales/olist_geolocation_dataset.csv  (61.3 MB)
✔ data/raw/sales/product_category_name_translation.csv  (0.0 MB)

All files ready, go to notebook 01.
